In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

required_env = [
    "DATABASE_URL",
    "OLLAMA_BASE_URL",
    "OLLAMA_MODEL",
]

print("Load environment variables:")
for key in required_env:
    print(f"- {key}: {'OK' if os.getenv(key) else 'MISSING'}")
    

Load environment variables:
- DATABASE_URL: OK
- OLLAMA_BASE_URL: OK
- OLLAMA_MODEL: OK


In [2]:
missing = [k for k in ["DATABASE_URL", "OLLAMA_BASE_URL", "OLLAMA_MODEL"] if not os.getenv(k)]
if missing:
    raise ReturnError(f"Missing required env vars: {missing}")

print("Environment check passed.")

Environment check passed.


In [3]:
from sqlalchemy import create_engine, text 

DATABASE_URL = os.environ["DATABASE_URL"]
engine = create_engine(DATABASE_URL, future =True)

with engine.connect() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
    version = conn.execute(text("SELECT version();")).scalar_one()
    conn.commit()

print("Database connected.")
print(version)
print("pgvector extension ensured.")

Database connected.
PostgreSQL 16.14 (Debian 16.14-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit
pgvector extension ensured.


In [4]:
from sqlalchemy import text

with engine.connect() as conn:
    value = conn.execute(text("SELECT 1;")).scalar_one()

print("SELECT 1 result:", value)

SELECT 1 result: 1


In [5]:
import json
import urllib.request

base_url = os.environ["OLLAMA_BASE_URL"].rstrip("/")
url = f"{base_url}/api/tags"

with urllib.request.urlopen(url, timeout=10) as resp:
    data = json.loads(resp.read().decode("utf-8"))
print("Ollama reachable")
print("Available models:", [m.get("name") for m in data.get("models", [])][:10])

Ollama reachable
Available models: ['llama3:8b']


In [6]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=os.environ["OLLAMA_MODEL"],
    base_url=os.environ["OLLAMA_BASE_URL"],
    temperature=0,
)

response = llm.invoke("Reply with exactly: setup-ok")
print(response.content)

setup-ok
